# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook demonstrates loading and interactive exploration of a dataset managed under the FAIR^2 (Findable, Accessible, Interoperable, Reusable) paradigm using the `mlcroissant` library. 

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. This will fetch and interpret the Croissant schema.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema JSON-LD URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset via mlcroissant
dataset = mlc.Dataset(croissant_url)

# Show metadata (not as a dictionary)
print(f"Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")

## 2. Data Overview
Review available record sets and fields. Refer to Croissant entities by their `@id` for consistent, schema-aligned access.

Let's inspect the record sets and a sample of their fields using their `@id`.

In [ ]:
# List available record sets and their field @ids
record_sets = list(dataset.metadata.record_sets)
print("Available record sets:")
for rs in record_sets:
    print(f"  RecordSet name: {rs.name}")
    print(f"    @id: {rs.id}")
    print("    Fields:")
    for f in rs.fields:
        print(f"      - {f.name} (@id: {f.id}, type: {f.data_type})")
    print()

## 3. Data Extraction
Load data from a chosen record set directly into a DataFrame. This process uses the Croissant record set `@id` for reference.

*Note*: Replace variables below with your chosen record set and field `@id`s from the overview. For demonstration, we use the **first** available record set.

In [ ]:
# Pick the first record set for this demonstration
record_set = record_sets[0]
record_set_id = record_set.id  # Store the @id for variable-driven access

# Load data as a pandas DataFrame
records = list(dataset.records(record_set=record_set_id))
df = pd.DataFrame(records)
print(f"Available columns in record set '{record_set.name}' (id: {record_set_id}):")
print(list(df.columns))

df.head()

## 4. Exploratory Data Analysis (EDA)
Filter, transform, and group columns from the DataFrame. All access to columns should be by their Croissant field `@id`.

For demonstration, we'll:
- Select a numeric field (the first with integer/float data type returned from Croissant metadata)
- Filter records with values greater than a threshold
- Normalize the filtered column
- Optionally group by a categorical column (if available)

In [ ]:
# Identify a numeric field (@id) from this record set
numeric_field = None
for field in record_set.fields:
    if field.data_type.lower() in ('integer', 'float', 'number'):
        numeric_field = field.id
        break

if numeric_field is None:
    print("No numeric field detected in this record set.")
else:
    print(f"Using numeric field '@id': {numeric_field}")

    # Convert column to numeric, forcing errors to NaN
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

    # Remove NaN values for demonstration
    filtered_df = df.dropna(subset=[numeric_field])

    # Use 10th percentile value as threshold for demonstration
    threshold = filtered_df[numeric_field].quantile(0.1)
    filtered_df = filtered_df[filtered_df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize
    mean = filtered_df[numeric_field].mean()
    std = filtered_df[numeric_field].std()
    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - mean) / std
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, norm_col]].head())

    # Optionally group by a categorical field (first with 'Text' or 'String' type)
    group_field = None
    for field in record_set.fields:
        if field.data_type.lower() in ('text', 'string'):
            group_field = field.id
            break
    if group_field:
        print(f"\nGrouping by field '@id': {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field and optionally its grouping by a categorical field.

*All fields referenced by their Croissant `@id`.*

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[numeric_field], kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

    if group_field:
        plt.figure(figsize=(10, 4))
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
        plt.title(f"Mean {numeric_field} grouped by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion

- This notebook demonstrated how to navigate, access, and analyze a Croissant-encoded, FAIR^2 dataset using the `mlcroissant` library.
- All datasets, record sets, and fields were referenced directly by their Croissant `@id` to ensure reproducibility and true semantic referencing.
- Further data cleaning and analytic procedures can be built similarly by querying the Croissant schema and referring to entity `@id`s.

For more advanced usage and available helpers, see [`mlcroissant` documentation](https://mlcommons.github.io/croissant/api/python/).